# Model 1 — All 115 Features + Random Forest (GLOBAL — All 9 Devices) — FIXED

Trains one Random Forest on all 115 features using data from **all 9 IoT devices**.

### What was fixed
1. **Memory bug in sampling**: the old version kept every chunk of every class in RAM and sampled only at the end, so the whole ~5 GB was loaded (swapping = hours). Now each class is **capped inside every chunk**, so memory stays small.
2. **float32** everywhere (half the RAM of float64).
3. **Progress printing** at every stage so you can see it is alive.
4. **Cached sample** (`global_train_sample.joblib`) so a crash never costs you the loading time again.
5. **Quick-test switch** (`QUICK_TEST`) to verify the pipeline on 1 device first.
6. Evaluation uses `usecols`, larger chunks and per-chunk progress.

In [ ]:
from pathlib import Path
import json, time, warnings, gc
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")

# ===== CONFIGURATION =====
PROJECT_ROOT = Path(r'c:\Users\kulsu\Downloads\detection+of+iot+botnet+attacks+n+baiot')
PREPROCESSED = PROJECT_ROOT / 'preprocessed'

ALL_DEVICES = [
    'Danmini_Doorbell',
    'Ecobee_Thermostat',
    'Ennio_Doorbell',
    'Philips_B120N10_Baby_Monitor',
    'Provision_PT_737E_Security_Camera',
    'Provision_PT_838_Security_Camera',
    'Samsung_SNH_1011_N_Webcam',
    'SimpleHome_XCS7_1002_WHT_Security_Camera',
    'SimpleHome_XCS7_1003_WHT_Security_Camera',
]

# ---- QUICK TEST: set True first to check everything works on 1 device (minutes) ----
QUICK_TEST = False
DEVICES = ALL_DEVICES[:1] if QUICK_TEST else ALL_DEVICES

TRAIN_CHUNK_SIZE = 100_000     # lower to 50_000 if you have <= 8 GB RAM
TEST_CHUNK_SIZE  = 200_000
MAX_PER_CLASS_PER_DEVICE = 2_000   # 2K x 11 classes x 9 devices ~ 200K rows max
RANDOM_STATE = 42
USE_CACHE = True

# Same RF parameters for ALL 4 models (fair comparison)
RF_PARAMS = dict(
    n_estimators=200,
    max_depth=25,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print(f'Devices: {len(DEVICES)} (QUICK_TEST={QUICK_TEST})')
print(f'Max rows per class per device: {MAX_PER_CLASS_PER_DEVICE}')

In [ ]:
# Identify feature columns (consistent across all devices)
sample_header = pd.read_csv(PREPROCESSED / ALL_DEVICES[0] / 'train.csv', nrows=0)
feature_cols = [c for c in sample_header.columns if c not in {'label', 'label_id'}]
FEAT_DTYPES = {c: 'float32' for c in feature_cols}
print(f'Feature count: {len(feature_cols)} (expected 115)')
print('Using text label column for global encoding (not per-device label_id)')

---
## 1. Load & Sample Training Data from ALL Devices

In [ ]:
def sample_device_train(device_name, feat_cols, max_per_class, seed=42):
    """Read one device's train.csv in chunks; cap each class DURING reading
    so memory never grows beyond ~max_per_class rows per class (x a few chunks)."""
    rng = np.random.default_rng(seed)
    train_path = PREPROCESSED / device_name / 'train.csv'
    kept = {}          # label -> list of DataFrames
    kept_n = {}        # label -> rows currently held

    for i, chunk in enumerate(pd.read_csv(train_path, chunksize=TRAIN_CHUNK_SIZE,
                                          usecols=feat_cols + ['label'],
                                          dtype=FEAT_DTYPES)):
        chunk = chunk.dropna(subset=['label'])
        for lbl, g in chunk.groupby('label'):
            if len(g) > max_per_class:                       # cap early
                g = g.sample(n=max_per_class,
                             random_state=int(rng.integers(1_000_000_000)))
            kept.setdefault(lbl, []).append(g)
            kept_n[lbl] = kept_n.get(lbl, 0) + len(g)
            # compress when a class holds far more than needed
            if kept_n[lbl] > 3 * max_per_class:
                merged = pd.concat(kept[lbl], ignore_index=True)
                merged = merged.sample(n=max_per_class,
                                       random_state=int(rng.integers(1_000_000_000)))
                kept[lbl] = [merged]
                kept_n[lbl] = len(merged)
        if i % 10 == 0:
            print(f'      chunk {i} read', flush=True)

    sampled = []
    for lbl in sorted(kept):
        df_cls = pd.concat(kept[lbl], ignore_index=True)
        if len(df_cls) > max_per_class:                      # final cap
            df_cls = df_cls.sample(n=max_per_class, random_state=seed)
        sampled.append(df_cls)

    del kept
    gc.collect()
    return pd.concat(sampled, ignore_index=True)


cache_file = PREPROCESSED / ('global_train_sample_quick.joblib' if QUICK_TEST
                             else 'global_train_sample.joblib')

t0 = time.time()
if USE_CACHE and cache_file.exists():
    print(f'Loading cached training sample: {cache_file.name}')
    df_train_global = joblib.load(cache_file)
else:
    all_train_parts = []
    for n, dev in enumerate(DEVICES, 1):
        t_dev = time.time()
        print(f'  [{n}/{len(DEVICES)}] Loading {dev}...', flush=True)
        df_part = sample_device_train(dev, feature_cols, MAX_PER_CLASS_PER_DEVICE, RANDOM_STATE)
        df_part['device'] = dev
        print(f'    -> {len(df_part):,} rows | Classes: {df_part["label"].nunique()} '
              f'| {time.time()-t_dev:.0f}s', flush=True)
        all_train_parts.append(df_part)

    df_train_global = pd.concat(all_train_parts, ignore_index=True)
    del all_train_parts
    gc.collect()
    if USE_CACHE:
        joblib.dump(df_train_global, cache_file)
        print(f'Cached sample to {cache_file.name}')

print(f'\nGlobal training set ready in {time.time()-t0:.1f}s')
print(f'Shape: {df_train_global.shape}')
print(f'Total rows: {len(df_train_global):,}')
print('\nClass distribution (global):')
print(df_train_global['label'].value_counts().sort_index())

In [ ]:
# Create a GLOBAL label encoding (consistent across all devices)
le = LabelEncoder()
le.fit(df_train_global['label'])

global_label_map = {label: int(idx) for idx, label in enumerate(le.classes_)}
print('Global Label Mapping:')
for lbl, idx in global_label_map.items():
    count = (df_train_global['label'] == lbl).sum()
    print(f'  {idx:2d} = {lbl:25s} ({count:,} training rows)')

# Prepare X and y
X_train = df_train_global[feature_cols].apply(pd.to_numeric, errors='coerce')
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0).astype('float32')
y_train = le.transform(df_train_global['label'])

print(f'\nX_train: {X_train.shape}')
print(f'y_train: {y_train.shape} | Unique classes: {len(np.unique(y_train))}')

---
## 2. Train Random Forest (All 115 Features, All Devices)

In [ ]:
model = RandomForestClassifier(**RF_PARAMS)

print(f'Training Random Forest with {len(feature_cols)} features on {len(X_train):,} rows...', flush=True)
t0 = time.time()
model.fit(X_train, y_train)
train_time = time.time() - t0
print(f'Training complete in {train_time:.1f} seconds')

---
## 3. Evaluate on ALL Devices' Test Sets (Combined)

In [ ]:
def evaluate_all_devices(model, le, feat_cols, devices):
    """Evaluate across all devices' test sets, chunked, with progress output."""
    y_true_all, y_pred_all = [], []
    device_results = {}
    total_inference = 0.0
    class_set = set(le.classes_)

    for n, dev in enumerate(devices, 1):
        test_path = PREPROCESSED / dev / 'test.csv'
        y_true_dev, y_pred_dev = [], []
        t_dev = time.time()

        for i, chunk in enumerate(pd.read_csv(test_path, chunksize=TEST_CHUNK_SIZE,
                                              usecols=feat_cols + ['label'],
                                              dtype=FEAT_DTYPES)):
            chunk = chunk[chunk['label'].isin(class_set)]
            if len(chunk) == 0:
                continue

            y_true = le.transform(chunk['label'])
            X = chunk[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0).astype('float32')

            t0 = time.time()
            y_pred = model.predict(X)
            total_inference += time.time() - t0

            y_true_dev.append(y_true)
            y_pred_dev.append(y_pred)
            print(f'    [{n}/{len(devices)}] {dev} chunk {i} done ({time.time()-t_dev:.0f}s)', flush=True)

        yt = np.concatenate(y_true_dev)
        yp = np.concatenate(y_pred_dev)

        dev_acc = accuracy_score(yt, yp)
        device_results[dev] = {'accuracy': float(dev_acc), 'rows': int(len(yt))}
        print(f'  {dev:45s}  acc={dev_acc:.4f}  rows={len(yt):>8,}', flush=True)

        y_true_all.append(yt)
        y_pred_all.append(yp)

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)
    labels = sorted(set(y_true) | set(y_pred))

    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    p_wt,  r_wt,  f_wt,  _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

    benign_idx = global_label_map.get('benign', 0)
    benign_mask = (y_true == benign_idx)
    fpr = 1.0 - accuracy_score(y_true[benign_mask], y_pred[benign_mask]) if benign_mask.sum() > 0 else None

    target_names = [le.classes_[i] for i in labels]

    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_precision': float(p_mac), 'macro_recall': float(r_mac), 'macro_f1': float(f_mac),
        'weighted_precision': float(p_wt), 'weighted_recall': float(r_wt), 'weighted_f1': float(f_wt),
        'false_positive_rate': float(fpr) if fpr is not None else None,
        'total_inference_time': total_inference,
        'avg_inference_per_sample_ms': float(total_inference / len(y_true) * 1000),
        'test_rows': int(len(y_true)),
        'labels': [int(l) for l in labels],
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=labels),
        'classification_report': classification_report(
            y_true, y_pred, labels=labels, target_names=target_names, zero_division=0
        ),
        'per_device': device_results
    }

print(f'Evaluating on {len(DEVICES)} device test set(s)...', flush=True)
t0 = time.time()
metrics = evaluate_all_devices(model, le, feature_cols, DEVICES)
eval_time = time.time() - t0

print(f"\n{'='*55}")
print(f'  MODEL 1: ALL FEATURES — GLOBAL ({len(DEVICES)} DEVICES)')
print(f"{'='*55}")
print(f'  Features used   : {len(feature_cols)}')
print(f'  Accuracy        : {metrics["accuracy"]:.4f}')
print(f'  Macro Precision : {metrics["macro_precision"]:.4f}')
print(f'  Macro Recall    : {metrics["macro_recall"]:.4f}')
print(f'  Macro F1-Score  : {metrics["macro_f1"]:.4f}')
fpr_val = metrics["false_positive_rate"]
print(f'  FPR (benign)    : {fpr_val:.4f}' if fpr_val is not None else '  FPR (benign)    : n/a')
print(f'  Inference time  : {metrics["total_inference_time"]:.2f}s')
print(f'  Total test rows : {metrics["test_rows"]:,}')
print(f"{'='*55}")

In [ ]:
print('Full Classification Report:\n')
print(metrics['classification_report'])

---
## 4. Save All Outputs

In [ ]:
output_dir = PREPROCESSED / 'global_models' / ('Model1_AllFeatures_QUICKTEST' if QUICK_TEST else 'Model1_AllFeatures')
output_dir.mkdir(parents=True, exist_ok=True)

# Save model
joblib.dump(model, output_dir / 'random_forest.joblib')

# Save label encoder
joblib.dump(le, output_dir / 'label_encoder.joblib')

# Save global label mapping
with open(output_dir / 'global_label_mapping.json', 'w') as f:
    json.dump({
        'label_to_id': global_label_map,
        'id_to_label': {v: k for k, v in global_label_map.items()}
    }, f, indent=2)

# Save feature list
with open(output_dir / 'selected_features.json', 'w') as f:
    json.dump({'method': 'all_features', 'count': len(feature_cols), 'features': feature_cols}, f, indent=2)

# Save metrics
metrics_save = {k: v for k, v in metrics.items() if k not in {'confusion_matrix', 'classification_report'}}
metrics_save.update({
    'model': 'Model1_AllFeatures',
    'scope': 'global_all_devices' if not QUICK_TEST else 'quick_test_1_device',
    'devices': DEVICES,
    'feature_selection': 'none',
    'feature_count': len(feature_cols),
    'training_rows': int(len(X_train)),
    'training_time_s': round(train_time, 2),
    'evaluation_time_s': round(eval_time, 2),
    'rf_params': {k: str(v) for k, v in RF_PARAMS.items()}
})
with open(output_dir / 'metrics.json', 'w') as f:
    json.dump(metrics_save, f, indent=2)

# Confusion matrix
target_names = [le.classes_[i] for i in metrics['labels']]
cm_df = pd.DataFrame(metrics['confusion_matrix'],
    index=[f'actual_{n}' for n in target_names],
    columns=[f'pred_{n}' for n in target_names])
cm_df.to_csv(output_dir / 'confusion_matrix.csv')

with open(output_dir / 'classification_report.txt', 'w') as f:
    f.write(metrics['classification_report'])

# Per-device accuracy table
dev_df = pd.DataFrame([
    {'Device': d, 'Accuracy': v['accuracy'], 'Test_Rows': v['rows']}
    for d, v in metrics['per_device'].items()
])
dev_df.to_csv(output_dir / 'per_device_accuracy.csv', index=False)

print(f'All outputs saved to: {output_dir}')
for f in sorted(output_dir.iterdir()):
    print(f'   {f.name}')
print('\nNext: Run Model 2 (Variance Threshold) global notebook')